# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR² Mental Health Survey Dataset](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library and the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset title:", metadata.name)
print("Description:", metadata.description)

## 2. Data Overview
Let's review the available record sets in this dataset, along with their `@id`s, fields, and columns. All references will use their `@id`.

> NOTE: Entities in mlcroissant are referenced and accessed using their `@id` as required.

First, let's enumerate the available record sets in the dataset. Then for each record set, we'll display its available fields and column mappings.

In [ ]:
# List all RecordSets in the dataset (access by @id)
record_sets = dataset.record_sets
print("Available RecordSets:")
for rs in record_sets:
    print(f"  @id: {rs.id} | Name: {rs.name}")

# For each RecordSet, show its fields
print("\nRecord Set Fields and Columns (@id):")
for rs in record_sets:
    print(f"\nRecordSet @id: {rs.id}, Name: {rs.name}")
    for field in rs.fields:
        col_ids = [col.id for col in getattr(field, 'columns', [])]
        print(f"  Field @id: {field.id}, Name: {field.name}, Columns: {col_ids}")

## 3. Data Extraction
We'll now extract data from one or more record sets into Pandas DataFrames for analysis. Each item is referenced strictly by its `@id`.

> Note: Replace or add record set `@id`s as desired based on the overview above. We'll extract the first record set below for demonstration.

In [ ]:
# List all record set @id's
record_set_ids = [rs.id for rs in dataset.record_sets]

print("Using these record set @id's to extract data:", record_set_ids)

dataframes = {}

# Extract records for each record set (by @id)
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Extracted DataFrame for record set {record_set_id}: shape = {df.shape}")

# Preview the columns of the first record set
main_record_set_id = record_set_ids[0]
print(f"\nDataFrame columns for {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's perform some common data processing steps on the main record set, such as filtering rows based on a numeric field, normalization, and grouping. All field access is by `@id` as listed previously.

> For demonstration, we'll pick `phq9_total_score` as a numeric field (see list in previous section), group by `village`, and show basic EDA. Please substitute `@id` values as they appear in your schema if different.

In [ ]:
# Example: use explicit @id values (replace with actual ones from overview if needed)
numeric_field_id = 'phq9_total_score'  # Example field @id for PHQ-9 total score
group_field_id = 'village'             # Example field @id for grouping

df = dataframes[main_record_set_id]

# Ensure these fields are numeric
if numeric_field_id in df.columns:
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df[[numeric_field_id, group_field_id]].head())

# Normalize the numeric field
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, norm_col]].head())

# Group by village and show mean PHQ-9 score if grouping column exists
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization

We'll visualize the distribution of PHQ-9 scores and the mean scores by village, using matplotlib. Make sure you have matplotlib installed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram of the PHQ-9 total scores
plt.figure(figsize=(8, 4))
df[numeric_field_id].dropna().hist(bins=20)
plt.title("Distribution of PHQ-9 Total Scores")
plt.xlabel("PHQ-9 Total Score")
plt.ylabel("Count")
plt.show()

# Barplot of mean PHQ-9 score by village (if available)
if group_field_id in df.columns:
    mean_scores = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    plt.figure(figsize=(10, 5))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=mean_scores)
    plt.title("Mean PHQ-9 Total Score by Village")
    plt.ylabel("Mean PHQ-9 Score")
    plt.xlabel("Village")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to:
- Load FAIR² survey metadata and records using the `mlcroissant` library.
- Explore dataset structure, referencing all entities by their `@id`.
- Extract and analyze demographic and mental health indicators (e.g., PHQ-9 scores).
- Perform preliminary EDA and visualization.

This dataset is suitable for in-depth mental health and demographic analysis, and the Croissant schema makes AI-ready data exploration reproducible and well-documented.

For further analyses, continue by exploring additional record sets, running machine learning pipelines, or linking with other domain ontologies (using `@id` for all entity references).